# DINOv3 — Chinee apple weed detection (Colab)

Self-contained Colab notebook for the `dinov3-colab` experiment in `weed-detection-experiments`.

What it does:
1. Clones the official [facebookresearch/dinov3](https://github.com/facebookresearch/dinov3) repo.
2. Loads DINOv3 weights from your Google Drive.
3. **Unsupervised (§4–6):** extracts per-patch features and finds the foreground via [CLS]-token saliency, with a Gradio UI to box the dominant object. Backbone-only — no class label.
4. **Supervised linear probe (§7–8):** trains a logistic-regression head on the *frozen* backbone from your labeled crops to classify Chinee apple vs other trees, saves it to Drive, and applies it per-patch for class-aware detection heatmaps.

**Before running:** Runtime → Change runtime type → **GPU** (T4 free is fine).

**One-time:** accept the DINOv3 license at https://ai.meta.com/resources/models-and-libraries/dinov3-downloads/ and place the `.pth` in your Drive (default expected path is shown in step 2).

**Note:** the `.pth` is the finished self-supervised backbone — it is *never* retrained. §7 only fits a small head on labeled crops; no pre-training images are needed from you.

## 0. Check the GPU

In [ ]:
!nvidia-smi

## 1. Mount Google Drive

Weights are read from Drive so you don't re-upload them each session.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 2. Configuration

Adjust paths if your Drive layout differs.

In [ ]:
import os

DINOV3_REPO = '/content/dinov3'
WEIGHTS_PATH = '/content/drive/MyDrive/dinov3/weights/dinov3_vitb16_pretrain_lvd1689m.pth'
ARCH = 'dinov3_vitb16'

assert os.path.isfile(WEIGHTS_PATH), f'Weights not found at {WEIGHTS_PATH} — update WEIGHTS_PATH or copy the .pth into Drive.'
print('Weights OK:', WEIGHTS_PATH)

## 3. Clone DINOv3 and install dependencies

In [ ]:
if not os.path.isdir(DINOV3_REPO):
    !git clone --depth 1 https://github.com/facebookresearch/dinov3.git {DINOV3_REPO}

%pip install -q pillow scipy scikit-learn torchmetrics 'gradio>=4.0'

## 4. Inference code

Same logic as `dinov3_detect.py`, inlined so the notebook is self-contained.

In [ ]:
import numpy as np
import torch
from PIL import Image, ImageDraw
from scipy.ndimage import find_objects, label
from torchvision import transforms

PATCH = 16
IMG_SIZE = 768
IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD = (0.229, 0.224, 0.225)

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Device:', DEVICE)

model = torch.hub.load(DINOV3_REPO, ARCH, source='local', weights=WEIGHTS_PATH).to(DEVICE).eval()

_tx = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

@torch.inference_mode()
def cls_saliency(image):
    # Cosine similarity between [CLS] and each patch token.
    # Higher = more aligned with the image's global object representation.
    x = _tx(image.convert('RGB')).unsqueeze(0).to(DEVICE)
    out = model.forward_features(x)
    cls = out['x_norm_clstoken'][0]
    patches = out['x_norm_patchtokens'][0]
    sim = torch.nn.functional.cosine_similarity(patches, cls.unsqueeze(0), dim=-1)
    grid = IMG_SIZE // PATCH
    return sim.float().cpu().numpy().reshape(grid, grid)

def foreground_mask(sim):
    s = (sim - sim.min()) / (sim.max() - sim.min() + 1e-8)
    return (s > s.mean()).astype(np.uint8)

def largest_component_bbox(mask):
    lbl, n = label(mask)
    if n == 0:
        return None
    sizes = np.bincount(lbl.ravel())
    sizes[0] = 0
    idx = int(sizes.argmax())
    sl = find_objects(lbl == idx)[0]
    return sl[1].start, sl[0].start, sl[1].stop, sl[0].stop

def annotate(image, bbox_grid, grid_size):
    if bbox_grid is None:
        return image
    W, H = image.size
    sx, sy = W / grid_size, H / grid_size
    x0, y0, x1, y1 = bbox_grid
    out = image.copy()
    ImageDraw.Draw(out).rectangle([x0 * sx, y0 * sy, x1 * sx, y1 * sy], outline='red', width=4)
    return out

def infer(image):
    sim = cls_saliency(image)
    bbox = largest_component_bbox(foreground_mask(sim))
    return annotate(image, bbox, sim.shape[0])

## 5. Debug — saliency heatmap

Upload an image to see what the model considers foreground. Useful when the Gradio output puts the box on the wrong region.

Panels: input | raw [CLS] saliency | overlay | mask + bbox.


In [ ]:
import io
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
from google.colab import files

uploaded = files.upload()
filename = next(iter(uploaded))
img = Image.open(io.BytesIO(uploaded[filename])).convert('RGB')

sim = cls_saliency(img)
mask = foreground_mask(sim)
bbox_grid = largest_component_bbox(mask)

sim_norm = (sim - sim.min()) / (sim.max() - sim.min() + 1e-8)
sim_big = Image.fromarray((sim_norm * 255).astype(np.uint8)).resize(img.size, Image.BILINEAR)
mask_big = Image.fromarray((mask * 255).astype(np.uint8)).resize(img.size, Image.NEAREST)

fig, axes = plt.subplots(1, 4, figsize=(20, 5))
axes[0].imshow(img); axes[0].set_title('Input'); axes[0].axis('off')
axes[1].imshow(sim_big, cmap='viridis'); axes[1].set_title('[CLS] saliency'); axes[1].axis('off')
axes[2].imshow(img); axes[2].imshow(sim_big, cmap='viridis', alpha=0.5); axes[2].set_title('Overlay'); axes[2].axis('off')
axes[3].imshow(img); axes[3].imshow(mask_big, cmap='Reds', alpha=0.4)
if bbox_grid is not None:
    W, H = img.size
    sx, sy = W / sim.shape[0], H / sim.shape[0]
    x0, y0, x1, y1 = bbox_grid
    axes[3].add_patch(plt.Rectangle((x0 * sx, y0 * sy), (x1 - x0) * sx, (y1 - y0) * sy, fill=False, edgecolor='red', linewidth=3))
axes[3].set_title('Mask + bbox'); axes[3].axis('off')
plt.tight_layout(); plt.show()

print(f'Saliency  min={sim.min():.3f}  max={sim.max():.3f}  mean={sim.mean():.3f}')
print(f'Mask     coverage={mask.mean() * 100:.1f}% of patches  ({mask.sum()} of {mask.size})')
print(f'Bbox     {bbox_grid}  (in patch grid; image is {img.size})')


## 6. Launch the Gradio UI

Click the public `*.gradio.live` URL to open the interface in a new tab.

In [ ]:
import gradio as gr

gr.Interface(
    fn=infer,
    inputs=gr.Image(type='pil', label='Upload'),
    outputs=gr.Image(type='pil', label='Detected object'),
    title='DINOv3 object localization — Chinee apple',
    description='Foreground is found via PCA over DINOv3 patch tokens; the largest connected blob is boxed. Backbone-only — no class label.',
).launch(share=True)

## 7. Supervised linear probe — classify Chinee apple vs other trees

The cells above are **backbone-only** (no class label — they just find "something"). This section
trains a tiny classifier **on top of the frozen DINOv3 features** so the model can actually name
Chinee apple and tell it apart from other trees.

**Nothing in DINOv3 is retrained.** We only fit a logistic-regression head on extracted features —
seconds on a GPU. You provide a small set of **labeled crops** (~30–50 per class to prototype,
~100–200 for a usable classifier), sorted into one folder per class. No pre-training, no boxes.

In [ ]:
# Drive folder with one subfolder per class, each holding cropped images:
#   dataset/
#   ├── chinee_apple/   crops centered on Chinee apple canopy
#   ├── other_tree/     crops of other trees/shrubs to distinguish from
#   └── background/     grass, soil, sky (optional, improves rejection)
DATASET_DIR = '/content/drive/MyDrive/dinov3/dataset'

CLASSES = sorted(
    d for d in os.listdir(DATASET_DIR)
    if os.path.isdir(os.path.join(DATASET_DIR, d))
)
print('Classes:', CLASSES)
for c in CLASSES:
    n = len([f for f in os.listdir(os.path.join(DATASET_DIR, c))
             if f.lower().endswith(('.jpg', '.jpeg', '.png'))])
    print(f'  {c}: {n} images')


In [ ]:
import numpy as np
from PIL import Image

@torch.inference_mode()
def patch_tokens(image):
    # (num_patches, dim) L2-normalized patch tokens for one image.
    x = _tx(image.convert('RGB')).unsqueeze(0).to(DEVICE)
    out = model.forward_features(x)
    p = out['x_norm_patchtokens'][0]
    return torch.nn.functional.normalize(p, dim=-1).float().cpu().numpy()

def image_embedding(image):
    # Mean-pooled patch tokens — SAME feature space used per-patch for the heatmap,
    # so the trained classifier transfers directly to dense (per-patch) inference.
    return patch_tokens(image).mean(0)

# Extract one embedding per labeled image.
X, y = [], []
for label_idx, c in enumerate(CLASSES):
    folder = os.path.join(DATASET_DIR, c)
    for f in sorted(os.listdir(folder)):
        if not f.lower().endswith(('.jpg', '.jpeg', '.png')):
            continue
        X.append(image_embedding(Image.open(os.path.join(folder, f))))
        y.append(label_idx)
X = np.stack(X); y = np.array(y)
print('Features:', X.shape, ' Labels:', y.shape, ' per class:', np.bincount(y))


In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
import matplotlib.pyplot as plt

X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.25, stratify=y, random_state=0)

# Frozen-feature linear probe. class_weight balances uneven folder sizes.
clf = LogisticRegression(max_iter=2000, C=1.0, class_weight='balanced')
clf.fit(X_tr, y_tr)

y_pred = clf.predict(X_te)
print(classification_report(y_te, y_pred, target_names=CLASSES))

cm = confusion_matrix(y_te, y_pred)
fig, ax = plt.subplots(figsize=(4, 4))
ax.imshow(cm, cmap='Blues')
ax.set_xticks(range(len(CLASSES))); ax.set_xticklabels(CLASSES, rotation=45, ha='right')
ax.set_yticks(range(len(CLASSES))); ax.set_yticklabels(CLASSES)
ax.set_xlabel('Predicted'); ax.set_ylabel('True')
for i in range(len(CLASSES)):
    for j in range(len(CLASSES)):
        ax.text(j, i, cm[i, j], ha='center', va='center')
plt.title('Confusion matrix'); plt.tight_layout(); plt.show()


### 7b. Save / reload the trained probe

Persists just the logistic-regression head + class names (~KB) to Drive. Reload it next session to
skip retraining — the frozen backbone (cells 1–10) is all you need to re-run alongside it.

In [ ]:
import joblib

# Persist the probe to Drive so you don't retrain each session.
# The backbone weights are NOT saved here — only the small head + class names.
PROBE_PATH = '/content/drive/MyDrive/dinov3/chinee_probe.joblib'

joblib.dump({'clf': clf, 'classes': CLASSES, 'img_size': IMG_SIZE, 'patch': PATCH}, PROBE_PATH)
print('Saved probe ->', PROBE_PATH)

# --- To reload in a later session, skip cells 16–18 and run this instead: ---
# bundle = joblib.load(PROBE_PATH)
# clf, CLASSES = bundle['clf'], bundle['classes']
# print('Loaded probe for classes:', CLASSES)


## 8. Class-aware detection — Chinee apple heatmap

The probe was trained on **patch-token features**, so we can run it on **every patch** of a new
image to get a per-class probability map. High-probability Chinee apple patches are boxed.

This turns the image-level classifier into a detector **with no extra training** — the same head,
applied densely. Upload an image to see: input | P(chinee apple) overlay | box.

In [ ]:
import io
from google.colab import files

# Index of the Chinee apple class in CLASSES (falls back to 0 if named differently).
CHINEE_IDX = CLASSES.index('chinee_apple') if 'chinee_apple' in CLASSES else 0

@torch.inference_mode()
def chinee_heatmap(image):
    # Apply the trained probe to EVERY patch token -> P(chinee apple) per patch.
    # Works because the probe was trained on mean-pooled patch tokens (same space).
    p = patch_tokens(image)
    prob = clf.predict_proba(p)[:, CHINEE_IDX]
    grid = IMG_SIZE // PATCH
    return prob.reshape(grid, grid)

uploaded = files.upload()
filename = next(iter(uploaded))
img = Image.open(io.BytesIO(uploaded[filename])).convert('RGB')

heat = chinee_heatmap(img)
pred = CLASSES[int(clf.predict(image_embedding(img)[None])[0])]
mask = (heat > 0.5).astype(np.uint8)
bbox_grid = largest_component_bbox(mask)

heat_big = Image.fromarray((heat * 255).astype(np.uint8)).resize(img.size, Image.BILINEAR)
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
axes[0].imshow(img); axes[0].set_title(f'Input — predicted: {pred}'); axes[0].axis('off')
axes[1].imshow(img); axes[1].imshow(heat_big, cmap='inferno', alpha=0.5)
axes[1].set_title('P(chinee apple) per patch'); axes[1].axis('off')
axes[2].imshow(img)
if bbox_grid is not None:
    W, H = img.size
    sx, sy = W / heat.shape[0], H / heat.shape[0]
    x0, y0, x1, y1 = bbox_grid
    axes[2].add_patch(plt.Rectangle((x0 * sx, y0 * sy), (x1 - x0) * sx, (y1 - y0) * sy, fill=False, edgecolor='lime', linewidth=3))
axes[2].set_title('Chinee apple box (p>0.5)'); axes[2].axis('off')
plt.tight_layout(); plt.show()
